# Анализ профиля заёмщика — версия 2.0

    **Цель:** Сравнить логистическую регрессию и случайный лес на табличных данных.

    **Алгоритм:** Предобработка обучается только на train; тестовые метрики учитывают дисбаланс классов. Это учебный пример, не решение о кредитовании.

    Учебная модернизация 2026-10-01. Код адаптирован с Python 3.7 на Python 3.12. Исходные алгоритмы сохранены; историческая
    версия доступна в Git на коммите `a842320`. Исправления учебных ошибок описаны
    в README. Это учебное пособие, не промышленная система и не benchmark.

    По умолчанию: CPU, короткий запуск, синтетические демонстрационные данные,
    без загрузок из сети. Реальные данные включаются отдельно: `MLLAB_DATA_SOURCE`.
    Синтетические результаты нельзя выдавать за качество на реальных данных.
    После изменения настроек перезапустите ядро и выполните ячейки сверху вниз.
    Окружение и структура данных описаны в `docs/ENVIRONMENTS.md` и `docs/DATA.md`.


In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get('MLLAB_ROOT', str(Path.cwd()))).resolve()
root = next((p for p in (start, *start.parents) if (p / 'ml_lab' / 'runtime.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Откройте папку репозитория или задайте MLLAB_ROOT.')
sys.path.insert(0, str(root))
from ml_lab.runtime import settings
cfg = settings('borrower-profile-analysis')


## Данные
Синтетическая выборка не содержит персональных данных; реальные CSV используются только локально.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix
target = 'SeriousDlqin2yrs'
if cfg.source == 'demo':
    x, y = make_classification(n_samples=480, n_features=10, n_informative=5, weights=[.85, .15], random_state=cfg.seed)
    frame = pd.DataFrame(x, columns=[f'feature_{i}' for i in range(x.shape[1])]); frame[target] = y
    train_frame, test_frame = train_test_split(frame, test_size=.25, stratify=y, random_state=cfg.seed)
elif cfg.source == 'local':
    train_frame = pd.read_csv(cfg.data / 'training_data.csv')
    test_frame = pd.read_csv(cfg.data / 'test_data.csv')
else:
    raise ValueError('Выберите demo или local')
for frame in [train_frame, test_frame]:
    if target not in frame or frame[target].isna().any():
        raise ValueError('Нужна известная непустая целевая колонка SeriousDlqin2yrs для учебной оценки.')
x_train, y_train = train_frame.drop(columns=[target]), train_frame[target]
x_test, y_test = test_frame.drop(columns=[target]), test_frame[target]
if set(x_train.columns) != set(x_test.columns): raise ValueError('Колонки train/test различаются.')
x_test = x_test[x_train.columns]
if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in x_train.dtypes):
    raise ValueError('Этот урок принимает числовые признаки; категории требуют отдельного encoder.')


## Pipeline и оценка
Удалена посторонняя ячейка с неопределённым df; проверка не скрывает предупреждения библиотек.


In [ ]:
models = {
    'Logistic regression': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=cfg.seed)),
    'Random forest': make_pipeline(SimpleImputer(strategy='median'), RandomForestClassifier(n_estimators=20 if cfg.quick else 200, class_weight='balanced', random_state=cfg.seed, n_jobs=1))}
scores = []
for name, model in models.items():
    model.fit(x_train, y_train)
    probabilities = model.predict_proba(x_test)[:, 1]
    prediction = model.predict(x_test)
    print(name, classification_report(y_test, prediction, zero_division=0))
    print('Confusion matrix:', confusion_matrix(y_test, prediction))
    scores.append({'model': name, 'ROC-AUC': roc_auc_score(y_test, probabilities), 'PR-AUC': average_precision_score(y_test, probabilities)})
result = pd.DataFrame(scores)
print(result)
result.to_csv(cfg.output / 'metrics.csv', index=False)
result.set_index('model').plot.bar(); plt.ylim(0, 1); plt.show()


## Результат и самостоятельная работа

    Выполненные ячейки показывают формы данных, обучение и результат алгоритма.
    Успешный короткий запуск проверяет работоспособность, а не достижение заданной точности.

    1. Почему accuracy недостаточно при дисбалансе?
2. Сравните ROC-AUC и PR-AUC.
3. Проверьте пропуски и scaler внутри Pipeline.
4. Объясните, почему целевую колонку нельзя заполнять средним.

    Перед выводом о качестве используйте отдельные train/validation/test,
    документируйте источник данных и сравнивайте эксперименты при одинаковых настройках.
